# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle containing 426K cars to understand what factors make a car more or less expensive. As a result of your analysis, you will provide clear recommendations to a used car dealership on inventory valuation.

### CRISP-DM Framework

To frame the task, we apply the Cross-Industry Standard Process for Data Mining (CRISP-DM).

### Business Understanding

#### Technical Problem Framing:
The business objective is reframed as a supervised continuous multiple regression task. Our target response variable is vehicle sale price ($y \in \mathbb{R}^+$). The feature space includes continuous attributes (odometer mileage, vehicle age) and categorical attributes (manufacturer, condition, cylinders, fuel, title status, transmission, drive, type). We evaluate regularized regression models (Ridge and Lasso) cross-validated via GridSearchCV to minimize predictive error while interpreting feature coefficients that drive market pricing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

### Data Understanding

Inspecting data dimensions, missing value proportions, and exploring continuous and categorical distributions.

In [ ]:
# Load dataset
try:
    df = pd.read_csv("../data/vehicles.csv")
except FileNotFoundError:
    try:
        df = pd.read_csv("data/vehicles.csv")
    except FileNotFoundError:
        df = pd.read_csv("vehicles.csv")

print(f"Initial shape: {df.shape}")
df.info()

In [ ]:
# Continuous feature distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample_price = df[(df["price"] > 1000) & (df["price"] < 80000)]["price"]
sns.histplot(sample_price, bins=40, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Vehicle Prices ($1,000 - $80,000)")
axes[0].set_xlabel("Price ($ USD)")
axes[0].set_ylabel("Count")

sample_odo = df[(df["odometer"] > 500) & (df["odometer"] < 250000)]["odometer"]
sns.histplot(sample_odo, bins=40, kde=True, ax=axes[1], color="#d95f02")
axes[1].set_title("Distribution of Odometer Readings (500 - 250,000 miles)")
axes[1].set_xlabel("Mileage (Miles)")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

### Data Preparation

Handling missing values, removing unrealistic outliers, engineering vehicle age, and constructing preprocessing transformers.

In [ ]:
# Filter unrealistic price and odometer values
df_clean = df.dropna(subset=["year", "odometer", "manufacturer"]).copy()
df_clean["year"] = df_clean["year"].astype(int)
df_clean = df_clean[
    (df_clean["price"] >= 1500) & (df_clean["price"] <= 80000) &
    (df_clean["odometer"] >= 500) & (df_clean["odometer"] <= 280000) &
    (df_clean["year"] >= 1995) & (df_clean["year"] <= 2022)
].copy()

df_clean["vehicle_age"] = 2024 - df_clean["year"]
drop_cols = ["id", "VIN", "url", "region_url", "image_url", "description", "posting_date", "county", "year"]
df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

cat_cols = ["condition", "cylinders", "fuel", "title_status", "transmission", "drive", "type"]
for c in cat_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna("unknown")

print(f"Cleaned records: {df_clean.shape[0]:,}")

In [ ]:
# Train/Test Split
sample_data = df_clean.sample(n=min(75000, len(df_clean)), random_state=42)
features = ["odometer", "vehicle_age", "condition", "cylinders", "fuel", "title_status", "transmission", "drive", "type"]
X = sample_data[features]
y = sample_data["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["odometer", "vehicle_age"]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ]
)

### Modeling

Building baseline Linear Regression alongside Ridge and Lasso models tuned via 5-fold cross-validation with GridSearchCV.

In [ ]:
# 1. Linear Regression
pipe_lr = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
pipe_lr.fit(X_train, y_train)

# 2. Ridge Regression
pipe_ridge = Pipeline([("preprocessor", preprocessor), ("regressor", Ridge())])
grid_ridge = GridSearchCV(pipe_ridge, {"regressor__alpha": [0.1, 1.0, 10.0, 100.0]}, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
grid_ridge.fit(X_train, y_train)

# 3. Lasso Regression
pipe_lasso = Pipeline([("preprocessor", preprocessor), ("regressor", Lasso(max_iter=3000, random_state=42))])
grid_lasso = GridSearchCV(pipe_lasso, {"regressor__alpha": [0.1, 1.0, 10.0, 50.0]}, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
grid_lasso.fit(X_train, y_train)

print("Tuned Ridge alpha:", grid_ridge.best_params_)
print("Tuned Lasso alpha:", grid_lasso.best_params_)

### Evaluation

Comparing model performance on unseen test data and establishing operational evaluation criteria.

In [ ]:
models = {
    "Linear Regression": pipe_lr,
    "Ridge Regression (Tuned)": grid_ridge.best_estimator_,
    "Lasso Regression (Tuned)": grid_lasso.best_estimator_
}

eval_records = []
for name, m in models.items():
    preds = m.predict(X_test)
    eval_records.append({
        "Model": name,
        "MAE ($)": mean_absolute_error(y_test, preds),
        "RMSE ($)": np.sqrt(mean_squared_error(y_test, preds)),
        "R2 Score": r2_score(y_test, preds)
    })

results_df = pd.DataFrame(eval_records)
display(results_df)

### Evaluation Metric Identification & Business Rationale

1. **Mean Absolute Error (MAE):** Selected as our primary operational metric for dealership stakeholders. Because it does not square errors, it preserves the natural dollar scale ($ USD). A dealership manager can directly interpret this as: *"On average, our valuation model is within $\pm \$X of the actual selling price."*
2. **Root Mean Squared Error (RMSE):** Used alongside MAE because it squares residuals, penalizing large mispricings heavily. This guarantees regularized models minimize extreme pricing errors on premium vehicles.
3. **Coefficient of Determination ($R^2$ Score):** Measures the percentage of vehicle price variance explained by the model features.

In [ ]:
# Coefficient Extraction from Optimal Lasso Model
best_lasso_reg = grid_lasso.best_estimator_.named_steps["regressor"]
encoded_cat_names = grid_lasso.best_estimator_.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(cat_cols)
all_features = num_cols + list(encoded_cat_names)

coef_df = pd.DataFrame({"Feature": all_features, "Coefficient": best_lasso_reg.coef_})
top_positive = coef_df[coef_df["Coefficient"] > 0].sort_values(by="Coefficient", ascending=False).head(8)
top_negative = coef_df[coef_df["Coefficient"] < 0].sort_values(by="Coefficient", ascending=True).head(8)
top_drivers = pd.concat([top_positive, top_negative]).sort_values(by="Coefficient").reset_index(drop=True)

plt.figure(figsize=(10, 6))
colors = ["#d95f02" if val < 0 else "#2b5c8f" for val in top_drivers["Coefficient"]]
sns.barplot(data=top_drivers, x="Coefficient", y="Feature", palette=colors, hue="Feature", legend=False)
plt.title("Top Value Drivers and Depreciation Factors (Lasso Regression)", fontsize=14)
plt.xlabel("Coefficient Impact on Price ($ USD)", fontsize=12)
plt.ylabel("Vehicle Feature", fontsize=12)
plt.tight_layout()
plt.show()

### Deployment & Executive Report for Dealership Inventory

#### 1. Inventory Sourcing Strategy
- **Trucks and Diesel Powertrains:** Diesel vehicles and pickup/truck body types command substantial resale premiums over time compared to sedans.
- **4WD/AWD Configurations:** Four-wheel and all-wheel drive models retain higher residual values across all age brackets.

#### 2. Risk & Depreciation Factors
- **Vehicle Age and Mileage:** Age and odometer readings are the primary continuous drivers of depreciation. Sourcing above 100,000 miles should require larger acquisition discounts.
- **Title Status:** Salvage or rebuilt titles produce the largest downward penalty on price, overwhelming other positive vehicle features.

#### 3. Recommended Next Steps
- Incorporate regional geographical segmentation into auction purchase rules.
- Integrate the regularized pricing model into lot valuation tools.